# EasyMagpieTTS — vLLM-Omni inference demo

This notebook runs an end-to-end inference pass through the
[`easymagpie_vllm_omni`](./easymagpie_vllm_omni) model definition using a
**converted checkpoint directory** produced by
[`easy_magpietts_convert_to_vllm.py`](./easy_magpietts_convert_to_vllm.py)
(weights + `config.json` + text tokenizer + speaker embeddings). It exercises the
full engine path: prefill -> autoregressive decode -> audio-code extraction.

It follows the same `AsyncOmni` single-stage pattern as the reference
`qwen3-tts` and `eartts` demos:

* **prefill** — the caller supplies the speaker-encoded context-audio embedding
  via `additional_information.speaker_embedding` `(T_audio, embedding_dim)` plus a
  plain `context_text` string; the model assembles the full prefill context
  (`[task_embedding? | speaker_embedding | context_text_embedded]`) and tokenizes
  `context_text` itself. `prompt_token_ids = [0] * prompt_len`, sized with
  `EasyMagpieTTSForConditionalGeneration.estimate_prompt_len(...)`.
* **decode** — each step consumes one subword id from the streaming
  `additional_information.text_tokens` list (the tokenized target sentence); the
  local transformer samples all `C * S` stacked audio codebooks for the frame.
* **output** — per-step audio codes are surfaced on
  `OmniOutput.multimodal_outputs[\"audio_codes\"]` (`BT x num_codebooks`), and the
  engine accumulates them across steps just like eartts, so we trim to the last
  `len(token_ids)` decoded rows.

> **Converted checkpoint.** Set `MODEL_DIR` below to the directory written by the
> converter. The engine reads the `config.json`, weights, and tokenizer straight
> from it — no hardcoded config, no dummy weights.

> **Environment.** Run this inside the bootstrapped `vllm_omni_env` (vLLM +
> vLLM-Omni + compatible torch) with the plugin installed:
> ```bash
> source /path/to/vllm_omni_env/bin/activate
> pip install -e examples/tts/easymagpie_vllm_omni
> ```

In [ ]:
import os

# Single-process executor below, but keep spawn semantics consistent with the
# qwen3-tts / eartts demos in case you switch to a multiproc backend.
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import json
import tempfile
import uuid
from pathlib import Path

import torch
import yaml

from vllm import SamplingParams
from vllm_omni import AsyncOmni

# Importing the model package is optional (the engine resolves the arch via the
# `vllm.general_plugins` entry point installed with the package), but doing it
# here surfaces the arch dataclass we use to read scalars from the config.
from easymagpie_vllm_omni.config import EasyMagpieOmniArch

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1. Point at the converted model directory

Set `MODEL_DIR` to the directory written by
[`easy_magpietts_convert_to_vllm.py`](./easy_magpietts_convert_to_vllm.py). It
already contains everything the engine needs:

* `config.json` — the registered arch + Nemotron-H backbone scalars + EasyMagpie
  scalars (read by `EasyMagpieOmniArch.from_hf_config`),
* `model.safetensors` — the converted weights (backbone + TTS submodules + the
  baked per-subword `text_embedding` table),
* the text-conditioning tokenizer (`tokenizer.json` / `tokenizer_config.json`),
  loaded in-engine to tokenize the per-request `context_text`,
* `speaker_embeddings/<name>.pt` — pre-computed speaker embeddings for reference
  voices.

The backbone is a **Nemotron-H** hybrid (Mamba2 + attention + MoE) decoder:
`EasyMagpieTTSForConditionalGeneration` constructs vLLM's `NemotronHModel` and
implements the hybrid-Mamba interfaces (`HasInnerState` / `IsHybrid` /
`SupportsMambaPrefixCaching`), exactly like the EasyMagpie vLLM *sidecar*. The
phoneme branch is enabled in the converted config; the model self-predicts
phonemes, so no phoneme stream needs to be supplied in the prompt.

We just read the `config.json` here to surface a few scalars used for building
the prompt (`text_vocab_size`, the audio EOS id, whether a task embedding exists).

In [ ]:
# Directory produced by easy_magpietts_convert_to_vllm.py.
MODEL_DIR = Path("/home/vklimkov/workspace/emp/NeMo/examples/tts/easymagpie_vllm_omni/easymp_vllm_model")
assert (MODEL_DIR / "config.json").exists(), f"No config.json under {MODEL_DIR}; run the converter first."

# Read the converted config to surface a few scalars used when building the
# prompt. The engine itself loads everything from MODEL_DIR; we only peek here.
config = json.loads((MODEL_DIR / "config.json").read_text())
arch = EasyMagpieOmniArch.from_hf_config(type("Cfg", (), config))

# Subword id space of the baked text-embedding table (the streaming text stream
# indexes into it). The model's text BOS/EOS/CFG-UNK ids are the last 3 rows.
TEXT_VOCAB = int(config["text_vocab_size"])
TEXT_EOS_ID = TEXT_VOCAB - 2  # matches EasyMagpieTTSInferenceModel.eos_id

print(f"Model dir                : {MODEL_DIR}")
print(f"embedding_dim            : {arch.embedding_dim}")
print(f"num_stacked_codebooks    : {arch.num_stacked_codebooks}  (C*S)")
print(f"tokens / codebook        : {arch.num_all_tokens_per_codebook}  (codebook_size + specials)")
print(f"audio_bos / audio_eos id : {arch.audio_bos_id} / {arch.audio_eos_id}")
print(f"text_vocab / text_eos    : {TEXT_VOCAB} / {TEXT_EOS_ID}")

## 2. Single-stage `AsyncOmni` engine

A single `llm` stage that runs the EasyMagpie talker, mirroring the eartts demo
(`worker_type=\"ar\"`, `OmniARScheduler`). The stage declares
`engine_output_type=\"audio\"` / `final_output_type=\"audio\"`: for a single-stage
AR TTS model these make the runner attach the per-step `audio_codes` multimodal
payload to the output (with `\"latent\"` the payload is dropped because nothing
downstream consumes it, and `multimodal_output[\"audio_codes\"]` comes back
`None`).

`skip_tokenizer_init: true` — we feed `prompt_token_ids` + `text_tokens`
directly, so vLLM doesn't need its own tokenizer for the prompt (the model still
loads the bundled `AutoTokenizer` from `MODEL_DIR` in-engine to tokenize
`context_text`).

`max_model_len` must cover `T_ctx` (prefill) + the number of decode steps.

In [ ]:
DECODE_STEPS = 256    # max number of audio frames to decode (trimmed at audio EOS)
# Prefill length is derived at prompt-build time from the speaker embedding +
# tokenized context_text (see the prompt cell); these just need to be large
# enough to cover prefill + decode.
MAX_MODEL_LEN = 1024
MAX_NUM_BATCHED_TOKENS = 1024

stage_cfg = {
    "stage_args": [
        {
            "stage_id": 0,
            "stage_type": "llm",
            "is_comprehension": True,
            "final_output": True,
            # "audio" (not "latent") is required for a single-stage AR TTS model:
            # it makes the AR model runner attach the per-step multimodal payload
            # ("audio_codes") to the EngineCoreOutput even though no downstream
            # stage consumes it, so the codes reach the client. With "latent" the
            # payload is dropped and multimodal_output["audio_codes"] is None.
            "final_output_type": "audio",
            "runtime": {"devices": "0"},
            "engine_args": {
                "model_stage": "easymagpie",
                "max_num_seqs": 1,
                "model_arch": "EasyMagpieTTSForConditionalGeneration",
                "worker_type": "ar",
                "scheduler_cls": "vllm_omni.core.sched.omni_ar_scheduler.OmniARScheduler",
                "enforce_eager": True,  # dummy run: skip CUDA-graph capture for a faster start
                "trust_remote_code": True,
                "async_scheduling": True,
                "enable_prefix_caching": False,
                "engine_output_type": "audio",
                "gpu_memory_utilization": 0.8,
                "distributed_executor_backend": "uni",
                "max_num_batched_tokens": MAX_NUM_BATCHED_TOKENS,
                "max_model_len": MAX_MODEL_LEN,
                # bf16 (not fp32): the Nemotron-H fused-MoE Triton kernel's block
                # sizes are tuned for 16-bit and overflow shared memory in fp32.
                #"dtype": "bfloat16",
                "dtype": "float16",
                "mamba_ssm_cache_dtype": "float32",
                "attention_backend": "TRITON_ATTN",
                # We feed prompt_token_ids + text_tokens directly; the model still
                # loads the bundled AutoTokenizer from MODEL_DIR for context_text.
                "skip_tokenizer_init": True,
            },
            "default_sampling_params": {
                "temperature": 0.0,
                "max_tokens": DECODE_STEPS,
                "detokenize": False,
                "ignore_eos": True,
            },
        }
    ],
}

_tmp = tempfile.NamedTemporaryFile(
    mode="w", suffix=".yaml", prefix="easymagpie_omni_demo_", delete=False,
)
yaml.dump(stage_cfg, _tmp, sort_keys=False)
_tmp.close()
STAGE_CFG_PATH = _tmp.name
print(f"Stage config: {STAGE_CFG_PATH}")

omni = AsyncOmni(
    model=str(MODEL_DIR),
    stage_configs_path=STAGE_CFG_PATH,
    log_stats=False,
    stage_init_timeout=300,
)
print("Engine ready (single stage: EasyMagpie talker)")

## 3. Build the prompt

Per-request input, passed through `additional_information`:

* **`speaker_embedding`** `(T_audio, embedding_dim)` — the speaker-encoded
  context-audio embedding (the audio branch of `prepare_context_tensors`),
  loaded here from `MODEL_DIR/speaker_embeddings/<SPEAKER_NAME>.pt` (written by
  the converter). The model assembles the full prefill context itself as
  `[task_embedding? | speaker_embedding | context_text_embedded]`.
* **`context_text`** — a plain conditioning string, here `"[EN]"`. The model
  tokenizes it in-engine and embeds it through the baked `text_embedding` table.
* **`text_tokens`** `list[int]` — the streaming subword stream: the target
  sentence tokenized with the bundled tokenizer, ending with the model's text
  EOS id. Decode step `k` consumes `text_tokens[k]`; once exhausted the channel
  is masked off (matching the reference `... encode(transcript) + [eos_id]`).

`prompt_token_ids = [0] * prompt_len` are placeholders (the model feeds the
backbone via `inputs_embeds`, never these ids). `prompt_len` must equal the
assembled context length, so we size it with the model's
`estimate_prompt_len(...)` — the length-only mirror of the in-engine prefill
assembly (à la qwen3-tts's `estimate_prompt_len_from_additional_information`).

(If the checkpoint had a phoneme branch you'd also stream `phoneme_tokens`.)

In [ ]:
torch.manual_seed(0)

from transformers import AutoTokenizer

from easymagpie_vllm_omni.easymagpie import EasyMagpieTTSForConditionalGeneration

# Speaker-encoded context audio (audio branch of prepare_context_tensors),
# pre-computed by the converter into MODEL_DIR/speaker_embeddings/<name>.pt.
SPEAKER_NAME = "eng"
_loaded = torch.load(MODEL_DIR / "speaker_embeddings" / f"{SPEAKER_NAME}.pt", map_location="cpu")
speaker_embedding = _loaded["speaker_encoding"] if isinstance(_loaded, dict) else _loaded
speaker_embedding = speaker_embedding.to(torch.float32)

# Plain conditioning string; the model tokenizes + embeds it in-engine.
CONTEXT_TEXT = "[EN]"

# Target sentence to synthesize.
TEXT = "Hello, this is a test of the EasyMagpie text to speech model."

# Same tokenizer the engine loads from MODEL_DIR. Used to (a) size the prefill
# placeholders so prompt_token_ids length matches the assembled context, and
# (b) tokenize the target sentence into the streaming text stream.
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
prompt_len = EasyMagpieTTSForConditionalGeneration.estimate_prompt_len(
    speaker_embedding,
    tokenize=lambda t: tokenizer.encode(t),
    context_text=CONTEXT_TEXT,
    has_task_embedding=arch.num_task_embeddings > 0,
)

# Streaming subword ids consumed one per decode step. Mirrors the reference
# `encode(transcript) + [eos_id]` (no BOS; HF special tokens disabled so the ids
# index the baked text_embedding table directly).
text_tokens = tokenizer.encode(TEXT, add_special_tokens=False) + [TEXT_EOS_ID]

# Audio (local-transformer) sampling params. vLLM's SamplingParams.temperature
# drives only the dummy backbone token sampler, so the *audio* temperature/top-k
# are forwarded via additional_information. temperature=0.0 == argmax
# (deterministic; matches the torch reference run with --temperature 0.0 --no_cfg).
LT_TEMPERATURE = 0.0
LT_TOPK = 80

additional_information = {
    "speaker_embedding": speaker_embedding,  # (T_audio, embedding_dim) tensor
    "context_text": CONTEXT_TEXT,            # plain string, tokenized in-model
    "text_tokens": text_tokens,              # list[int], grows by one per step
    "temperature": LT_TEMPERATURE,           # audio sampling temperature (local transformer)
    "top_k": LT_TOPK,                        # audio sampling top-k (local transformer)
}

prompt = {
    "prompt_token_ids": [0] * prompt_len,    # prefill placeholders
    "additional_information": additional_information,
}

assert prompt_len + DECODE_STEPS <= MAX_MODEL_LEN, (
    f"prompt_len ({prompt_len}) + decode steps ({DECODE_STEPS}) exceeds "
    f"MAX_MODEL_LEN ({MAX_MODEL_LEN}); raise MAX_MODEL_LEN / MAX_NUM_BATCHED_TOKENS."
)

print(f"speaker_embedding            : {tuple(speaker_embedding.shape)}")
print(f"context_text                 : {CONTEXT_TEXT!r} -> {tokenizer.encode(CONTEXT_TEXT)}")
print(f"text                         : {TEXT!r}")
print(f"text_tokens (len {len(text_tokens):3d})        : {text_tokens[:8]}{' ...' if len(text_tokens) > 8 else ''}")
print(f"prompt_len (placeholders)    : {prompt_len}")
print(f"decode steps (max_tokens)    : {DECODE_STEPS}")

sampling_params = SamplingParams(
    temperature=0.0,    # backbone token sampler is a no-op (audio is sampled in the local transformer)
    max_tokens=DECODE_STEPS,
    detokenize=False,
    ignore_eos=True,    # audio EOS lives in the codes, not the vLLM token stream -> run the budget + trim
)

## 4. Run inference and extract audio codes

`omni.generate(...)` is an async generator yielding one `RequestOutput` per
engine step; we keep the last one. As in the eartts demo, the accumulated
`multimodal_output[\"audio_codes\"]` holds one row per flat-batch token over the
whole run (the `T_ctx` prefill frames — codes left zero — plus one frame per
decode step), so we trim to the last `len(token_ids)` rows to recover just the
decoded frames, then trim again at the audio EOS frame (the model signals
end-of-speech in the codes, not in the vLLM token stream).

In [ ]:
async def run_request(prompt: dict, sampling_params):
    request_id = f"easymagpie-{uuid.uuid4().hex[:8]}"
    final_ro = None
    num_steps = 0
    async for stage_output in omni.generate(
        prompt,
        sampling_params_list=[sampling_params],
        request_id=request_id,
    ):
        final_ro = stage_output
        num_steps += 1
    return final_ro, num_steps


final_ro, num_steps = await run_request(prompt, sampling_params)
assert final_ro is not None, "no output from engine"

mm = final_ro.multimodal_output or {}
audio_codes = mm.get("audio_codes")
token_ids = final_ro.outputs[0].token_ids if final_ro.outputs else []

print(f"Engine steps yielded       : {num_steps}")
print(f"Layer-0 tokens (token_ids) : {len(token_ids)}")
if isinstance(audio_codes, torch.Tensor):
    audio_codes = audio_codes.detach().cpu().to(torch.long)
    print(f"audio_codes shape (raw)    : {tuple(audio_codes.shape)}")
    # Trim the Tref prefill frames echoed during prefill: keep only the decoded
    # frames (the last len(token_ids) rows), exactly like the eartts demo.
    if len(token_ids) > 0:
        audio_codes = audio_codes[-len(token_ids):].contiguous()

    # Drop the leading streaming_speech_delay warm-up frames. With the streaming
    # delay the audio stream only opens at decode step == speech_delay, so the
    # first speech_delay decoded frames carry no real audio (the audio channel was
    # masked off while the model consumed lookahead text/phonemes).
    speech_delay = int(getattr(arch, "streaming_speech_delay", 0) or 0)
    if speech_delay > 0:
        print(f"dropping {speech_delay} leading speech-delay warm-up frames")
        audio_codes = audio_codes[speech_delay:].contiguous()

    # Trim at the audio EOS: the model signals end-of-speech inside the codes
    # (codebook 0 == audio_eos_id), not via the vLLM token stream.
    eos_frames = (audio_codes[:, 0] == arch.audio_eos_id).nonzero(as_tuple=True)[0]
    if eos_frames.numel() > 0:
        eos_idx = int(eos_frames[0])
        print(f"audio EOS at frame         : {eos_idx} / {audio_codes.shape[0]}")
        audio_codes = audio_codes[:eos_idx].contiguous()
    else:
        print(f"no audio EOS within budget ({DECODE_STEPS} frames); using full decode")

    print(f"audio_codes shape (decode) : {tuple(audio_codes.shape)}")
    print(f"audio_codes dtype          : {audio_codes.dtype}")
    print(f"codes min / max            : {int(audio_codes.min())} / {int(audio_codes.max())}")
    print(f"first frame codes          : {audio_codes[0].tolist()}")
else:
    print(f"audio_codes                : {audio_codes!r}")

In [ ]:
import matplotlib.pylab as plt

plt.imshow(audio_codes.T, aspect="auto")
plt.colorbar()
plt.show()

## 5. Decode audio codes to a waveform

The engine emits **stacked** codebooks: `audio_codes` is `(T, C*S)` where
`C = num_audio_codebooks` and `S = frame_stacking_factor` (here `C*S = 16`).
To turn them back into a waveform we mirror what
[`EasyMagpieTTSInferenceModel.streaming_finalize`](../../../nemo/collections/tts/models/easy_magpietts_inference.py)
does at the end of inference:

1. **load the codec** — the `.nemo` audio codec used to train the model
   (`AudioCodecModel.restore_from(...)`, discriminator dropped to save memory),
2. **unstack** `(T, C*S)` -> `(1, C, T*S)` — the inverse of `stack_codes`,
3. **convert codec tokens** — this checkpoint was trained on a *regrouped* FSQ
   index space (8 codebooks of size 1024) that differs from the codec's native
   `GroupFiniteScalarQuantizer` (4 codebooks), so we map the model's tokens back
   to the codec's space (`convert_new_to_original`) before decoding. The
   `vector_quantizer` config is read straight from the source EasyMagpie `.nemo`
   (config only, no weights). If the two spaces match, this step is skipped.
4. **decode** `codec_model.decode(tokens=..., tokens_len=...)` -> waveform at
   `codec_model.output_sample_rate`.

> Set `CODEC_MODEL_PATH` / `EASYMAGPIE_NEMO` to the **same** `.nemo` files passed
> to `easy_magpietts_convert_to_vllm.py` (`--codec_model_path` / `--nemo_file`).
> This step needs NeMo importable in the current environment.

In [ ]:
from hydra.utils import instantiate
from IPython.display import Audio, display

from nemo.collections.tts.models import AudioCodecModel
from nemo.collections.tts.models.easy_magpietts_inference import EasyMagpieTTSInferenceModel
from nemo.collections.tts.modules.audio_codec_modules import VectorQuantizerIndexConverter

# Same .nemo codec passed to easy_magpietts_convert_to_vllm.py --codec_model_path.
CODEC_MODEL_PATH = "/home/vklimkov/workspace/emp/ckpt/easymagpietts_NEXT/25fps_spectral_codec_with_bandwidth_extension.nemo"

# --- load the codec once (drop the discriminator to save memory) ---
_codec_cfg = AudioCodecModel.restore_from(CODEC_MODEL_PATH, return_config=True)
if "use_scl_loss" in _codec_cfg:
    _codec_cfg.use_scl_loss = False
codec_model = AudioCodecModel.restore_from(
    CODEC_MODEL_PATH, strict=False, override_config_path=_codec_cfg
)
if hasattr(codec_model, "discriminator"):
    del codec_model.discriminator
codec_model = codec_model.eval().to("cuda" if torch.cuda.is_available() else "cpu")
codec_device = next(codec_model.parameters()).device

# Source EasyMagpie .nemo (--nemo_file). Only its config is read (no weights),
# to recover the `vector_quantizer` override the model was trained with.
EASYMAGPIE_NEMO = "/home/vklimkov/workspace/emp/ckpt/easymagpietts_NEXT/2605_NemotronTTS_V0.2/v2/2605_EMTTS_SmallMamba_Step150k_posttrained_epoch12.nemo"

# --- optional codec-token converter ---------------------------------------
# EasyMagpie can be trained on a *regrouped* FSQ index space (here C=8 codebooks
# of size 1024) that differs from the codec's native quantizer (this codec's
# GroupFiniteScalarQuantizer has 4 codebooks). When they differ the model's
# tokens must be mapped back to the codec's space before decoding, exactly as
# EasyMagpieTTSInferenceModel does via `_codec_converter` /
# `CodecHelper.codes_to_audio`.
_em_cfg = EasyMagpieTTSInferenceModel.restore_from(EASYMAGPIE_NEMO, return_config=True)
_vq_cfg = _em_cfg.get("vector_quantizer")
if _vq_cfg is not None and instantiate(_vq_cfg).num_codebooks != codec_model.vector_quantizer.num_codebooks:
    codec_converter = VectorQuantizerIndexConverter(
        vector_quantizer_original=codec_model.vector_quantizer,
        vector_quantizer_new=instantiate(_vq_cfg),
    ).to(codec_device)
else:
    codec_converter = None
print(f"codec native codebooks  : {codec_model.vector_quantizer.num_codebooks}")
print(f"codec token converter   : {'enabled' if codec_converter is not None else 'not needed'}")

S = arch.frame_stacking_factor                       # stacking factor (sub-frames per stacked frame)
C = arch.num_stacked_codebooks // S                  # real codec codebooks
assert audio_codes.dim() == 2 and audio_codes.size(1) == arch.num_stacked_codebooks, (
    f"expected audio_codes (T, {arch.num_stacked_codebooks}); got {tuple(audio_codes.shape)}"
)

# --- unstack (T, C*S) -> (1, C, T*S): inverse of EasyMagpie stack_codes ---
stacked = audio_codes.to(codec_device, torch.long).T.unsqueeze(0)   # (1, C*S, T)
T_out = stacked.size(-1)
codes = stacked.view(1, C, S, T_out).permute(0, 1, 3, 2).reshape(1, C, T_out * S)  # (1, C, T*S)
codes_len = torch.tensor([codes.size(-1)], device=codec_device, dtype=torch.long)

# Pad very short sequences (codec needs a few frames), matching _prepare_codes_for_decode.
MIN_LEN = 4
if int(codes_len.min()) < MIN_LEN:
    codes = torch.nn.functional.pad(codes, (0, MIN_LEN - int(codes_len.min())), value=0)
    codes_len = codes_len.clamp(min=MIN_LEN)

# Drop any stray special tokens (BOS/EOS/MASK live at codebook_size..) so every
# index is a valid codec entry before decoding.
codes = codes.clamp_(0, arch.codebook_size - 1)

# --- decode codes -> waveform (mirrors CodecHelper.codes_to_audio) ---
with torch.no_grad(), torch.autocast(device_type=codec_device.type, dtype=torch.float32):
    if codec_converter is not None:
        codes = codec_converter.convert_new_to_original(audio_tokens=codes, audio_lens=codes_len)
    audio, audio_len = codec_model.decode(tokens=codes, tokens_len=codes_len)

waveform = audio[0, : int(audio_len[0])].detach().cpu().float().numpy()
sample_rate = int(codec_model.output_sample_rate)

print(f"codes (unstacked) shape : {tuple(codes.shape)}  (1, C={C}, T*S={codes.size(-1)})")
print(f"waveform samples        : {waveform.shape[0]}  ({waveform.shape[0] / sample_rate:.2f}s @ {sample_rate} Hz)")

display(Audio(waveform, rate=sample_rate))